In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import pickle
from sklearn.preprocessing import MinMaxScaler

sys.path.append('../')
from third_party.HDCmodel import *
from conformalHDC.models import *
from conformalHDC.methods import *
from conformalHDC.utils import *

In [2]:
base_path="C:/Users/liang/Documents/GitHub/conformalHDC/data/CTG"

In [3]:
df = pd.read_excel(os.path.join(base_path,'CTG.xls'), sheet_name="Data", header=1)

In [4]:
# Define the columns to keep as features and the class label
feature_columns = [
    'LB', 'AC', 'FM', 'UC', 'DL', 'DS', 'DP', 'ASTV', 'MSTV', 'ALTV', 'MLTV',
    'Width', 'Min', 'Max', 'Nmax', 'Nzeros', 'Mode', 'Mean', 'Median', 'Variance', 'Tendency'
]
class_column = 'NSP'

In [5]:
df_selected = df[feature_columns + [class_column]]
df_selected = df_selected[:-3]

In [6]:
df_selected

,LB,AC,FM,UC,DL,DS,DP,ASTV,MSTV,ALTV,...,Min,Max,Nmax,Nzeros,Mode,Mean,Median,Variance,Tendency,NSP
0,120.0,0.0,0.0,0.0,0.0,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,4.0,0.0,4.0,2.0,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,2.0,0.0,5.0,2.0,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,2.0,0.0,6.0,2.0,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,4.0,0.0,5.0,0.0,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2121,140.0,0.0,0.0,6.0,0.0,0.0,0.0,79.0,0.2,25.0,...,137.0,177.0,4.0,0.0,153.0,150.0,152.0,2.0,0.0,2.0
2122,140.0,1.0,0.0,9.0,0.0,0.0,0.0,78.0,0.4,22.0,...,103.0,169.0,6.0,0.0,152.0,148.0,151.0,3.0,1.0,2.0
2123,140.0,1.0,0.0,7.0,0.0,0.0,0.0,79.0,0.4,20.0,...,103.0,170.0,5.0,0.0,153.0,148.0,152.0,4.0,1.0,2.0
2124,140.0,1.0,0.0,9.0,0.0,0.0,0.0,78.0,0.4,27.0,...,103.0,169.0,6.0,0.0,152.0,147.0,151.0,4.0,1.0,2.0


In [7]:
# Separate features and class label
X = df_selected.drop(columns=[class_column])
y = np.array(df_selected[class_column])

# Normalize all feature columns to [-1, 1]
scaler = MinMaxScaler(feature_range=(-1, 1))
X = scaler.fit_transform(X)
X = np.clip(X, -1.0, 1.0)  # Clip the values to ensure they 'fall within [-1, 1]
y[y==3]=2

In [10]:
df_combined = pd.DataFrame(X, columns=feature_columns)
df_combined['Label'] = y

In [11]:
df_combined

,LB,AC,FM,UC,DL,DS,DP,ASTV,MSTV,ALTV,...,Min,Max,Nmax,Nzeros,Mode,Mean,Median,Variance,Tendency,Label
0,-0.481481,-1.000000,-1.000000,-1.000000,-1.00,-1.0,-1.0,0.626667,-0.911765,-0.054945,...,-0.779817,-0.931034,-0.777778,-1.0,-0.055118,0.174312,-0.192661,-0.457249,1.0,2.0
1,-0.037037,-0.692308,-1.000000,-0.652174,-0.75,-1.0,-1.0,-0.866667,-0.441176,-1.000000,...,-0.669725,0.310345,-0.333333,-0.8,0.275591,0.155963,0.155963,-0.910781,0.0,1.0
2,0.000000,-0.846154,-1.000000,-0.565217,-0.75,-1.0,-1.0,-0.893333,-0.441176,-1.000000,...,-0.669725,0.310345,-0.444444,-0.8,0.275591,0.137615,0.119266,-0.903346,0.0,1.0
3,0.037037,-0.846154,-1.000000,-0.478261,-0.75,-1.0,-1.0,-0.893333,-0.352941,-1.000000,...,-0.944954,-0.172414,0.222222,-1.0,0.212598,0.119266,0.100917,-0.903346,1.0,1.0
4,-0.037037,-0.692308,-1.000000,-0.565217,-1.00,-1.0,-1.0,-0.893333,-0.352941,-1.000000,...,-0.944954,-0.172414,0.000000,-1.0,0.212598,0.155963,0.119266,-0.918216,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2121,0.259259,-1.000000,-1.000000,-0.478261,-1.00,-1.0,-1.0,0.786667,-1.000000,-0.450549,...,0.596330,-0.051724,-0.555556,-1.0,0.464567,0.412844,0.376147,-0.985130,0.0,2.0
2122,0.259259,-0.923077,-1.000000,-0.217391,-1.00,-1.0,-1.0,0.760000,-0.941176,-0.516484,...,-0.027523,-0.189655,-0.333333,-1.0,0.448819,0.376147,0.357798,-0.977695,1.0,2.0
2123,0.259259,-0.923077,-1.000000,-0.391304,-1.00,-1.0,-1.0,0.786667,-0.941176,-0.560440,...,-0.027523,-0.172414,-0.444444,-1.0,0.464567,0.376147,0.376147,-0.970260,1.0,2.0
2124,0.259259,-0.923077,-1.000000,-0.217391,-1.00,-1.0,-1.0,0.760000,-0.941176,-0.406593,...,-0.027523,-0.189655,-0.333333,-1.0,0.448819,0.357798,0.357798,-0.970260,1.0,2.0


In [12]:
# Save the combined DataFrame to a CSV file
df_combined.to_csv('cleaned_data.csv', index=False)